# CSE 151B — Submission notebook (`private.jsonl`)

Live in **`notebooks/submission.ipynb`**. Same inference stack as [`full_public.ipynb`](full_public.ipynb), run on all of **`data/private.jsonl`** (no labels).

1. (**Colab A100**) `%pip` → restart runtime → Drive copy cell.
2. Set **`MAX_TOKENS`** in §2 (default **32k**, matching pub-003).
3. Prompts are **per question**: baseline for MCQ and single-blank free-form; multi-blank format when `[ANS]` count > 1.
4. Generate with checkpointing → write **`results/submission.csv`**.

**Output:** CSV with columns `id`, `response` (full model trace, CSV-escaped). Checkpoint: `results/submission.responses.jsonl`.

In [ ]:
!git clone https://github.com/keguangzhang/151B_SP26_Competition.git
%cd 151B_SP26_Competition
!git checkout dev-andrew

Cloning into '151B_SP26_Competition'...
remote: Enumerating objects: 406, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 406 (delta 45), reused 27 (delta 27), pack-reused 351 (from 2)
Receiving objects: 100% (406/406), 5.75 MiB | 21.49 MiB/s, done.
Resolving deltas: 100% (204/204), done.
/content/151B_SP26_Competition
Branch 'dev-andrew' set up to track remote branch 'dev-andrew' from 'origin'.
Switched to a new branch 'dev-andrew'


In [ ]:

%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers "bitsandbytes>=0.48.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 73.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 73.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 256.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 142.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 250.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 249.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 194.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 MB 

In [ ]:
%pip install -q datasets peft trl accelerate
%pip install -q "bitsandbytes>=0.48.1"
%pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 48.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.6 MB/s eta 0:00:00


In [ ]:
%pip install -U pyarrow datasets

In [ ]:
# Read all the qestions and answer pairs in public.jsonl and convert into SFT training text


from datasets import Dataset
import json

PUBLIC_PATH = "/content/151B_SP26_Competition/data/public.jsonl"

rows = []
with open(PUBLIC_PATH, "r") as f:
    for line in f:
        rows.append(json.loads(line))

def make_sft_example(ex):
    question = ex["question"]

    answer = ex["answer"]
    if isinstance(answer, list):
        answer = ", ".join(str(a) for a in answer)
    else:
        answer = str(answer)

    text = f"""<|im_start|>system
You are an expert mathematician. Solve the problem and provide the final answer clearly.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
\\boxed{{{answer}}}
<|im_end|>"""

    return {"text": text}

sft_data = Dataset.from_list([make_sft_example(ex) for ex in rows])

print(sft_data[0]["text"])
print("Total examples:", len(sft_data))

from datasets import Dataset

rows = []
with open(PUBLIC_PATH, "r") as f:
    for line in f:
        rows.append(json.loads(line))

# Shuffle for randomness, split pairs in 900 training and 226 testing.
import random
random.seed(42)
random.shuffle(rows)

split_idx = int(0.8 * len(rows))

train_rows = rows[:split_idx]
valid_rows = rows[split_idx:]

print("Train:", len(train_rows))
print("Valid:", len(valid_rows))

train_dataset = Dataset.from_list(
    [make_sft_example(ex) for ex in train_rows]
)

valid_dataset = Dataset.from_list(
    [make_sft_example(ex) for ex in valid_rows]
)

<|im_start|>system
You are an expert mathematician. Solve the problem and provide the final answer clearly.
<|im_end|>
<|im_start|>user
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]
<|im_end|>
<|im_start|>assistant
\boxed{325*(1+325)}
<|im_end|>
Total examples: 1126
Train: 900
Valid: 226


In [ ]:
# Load model for QLoRA training

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"  # use the exact model id from your pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

In [ ]:
#loRA configuration

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

In [ ]:
#Trainig

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/151B_SP26_Competition/qwen3_lora_sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    max_length=2048,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.592392
20,1.085831
30,0.933987
40,0.943605
50,0.990213
60,0.918843
70,0.893273
80,0.971983
90,0.913485
100,0.947630


TrainOutput(global_step=113, training_loss=1.009184525076267, metrics={'train_runtime': 726.7668, 'train_samples_per_second': 1.238, 'train_steps_per_second': 0.155, 'total_flos': 3289042240527360.0, 'train_loss': 1.009184525076267, 'entropy': 0.9030417710542679, 'num_tokens': 149507.0, 'mean_token_accuracy': 0.7661805659532547, 'epoch': 1.0})

In [ ]:
trainer.save_model("/content/151B_SP26_Competition/qwen3_lora_sft")
tokenizer.save_pretrained("/content/151B_SP26_Competition/qwen3_lora_sft")

('/content/151B_SP26_Competition/qwen3_lora_sft/tokenizer_config.json',
 '/content/151B_SP26_Competition/qwen3_lora_sft/chat_template.jinja',
 '/content/151B_SP26_Competition/qwen3_lora_sft/tokenizer.json')

In [ ]:
import os
os.environ["VLLM_USE_V1"] = "0"
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
ADAPTER_PATH = "/content/drive/MyDrive/qwen3_lora_sft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

llm = LLM(
    model=MODEL_ID,
    enable_lora=True,
    max_model_len=4096,
    gpu_memory_utilization=0.75,
    trust_remote_code=True,
)

INFO 05-30 17:45:38 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enable_lora': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
WARNING 05-30 17:45:38 [envs.py:1818] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 05-30 17:45:38 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-30 17:45:38 [model.py:1680] Using max model len 4096
INFO 05-30 17:45:38 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
WARNING 05-30 17:45:38 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
ADAPTER_PATH = "/content/drive/MyDrive/qwen3_lora_sft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("LoRA adapter loaded!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LoRA adapter loaded!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!find /content -name "adapter_config.json" 2>/dev/null

/content/drive/MyDrive/qwen3_lora_sft/checkpoint-100/adapter_config.json
/content/drive/MyDrive/qwen3_lora_sft/checkpoint-113/adapter_config.json
/content/drive/MyDrive/qwen3_lora_sft/adapter_config.json


In [ ]:
from tqdm import tqdm
import torch

SYSTEM_PROMPT = (
    "You are an expert mathematician. "
    "Solve the problem carefully. "
    "Do not refuse. "
    "End your response with exactly one final answer in the format \\boxed{...}."
)

batch_size = 8
max_new_tokens = 256

valid_responses = []

for i in tqdm(range(0, len(eval_rows), batch_size), desc="Generating"):
    batch = eval_rows[i:i+batch_size]

    prompts = []

    for item in batch:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["question"]},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        prompts.append(prompt)

    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    for j, output in enumerate(outputs):
        generated_tokens = output[inputs["input_ids"].shape[1]:]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        valid_responses.append(response)

print("Generated:", len(valid_responses))
print(valid_responses[0])

Generating: 100%|██████████| 4/4 [01:57<00:00, 29.30s/it]

Generated: 30



In [ ]:
question = valid_rows[0]["question"]

messages = [
    {
        "role": "system",
        "content": (
            "You are an expert mathematician. Solve the problem carefully. "
            "Do not refuse. End your response with exactly one final answer in the format \\boxed{...}."
        )
    },
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=521,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

system
You are an expert mathematician. Solve the problem carefully. Do not refuse. End your response with exactly one final answer in the format \boxed{...}.
user
Use the properties of exponents to simplify the following
$38B^0=$ [ANS]
assistant
<think>
Okay, let's see. The problem is to simplify 38 times B to the power of 0. Hmm, I remember that any non-zero number raised to the power of 0 is 1. So, B^0 should be 1, right? Let me make sure. Yeah, the zero exponent rule says that any number (except zero) raised to the 0 power is 1. So even if B is a variable, as long as it's not zero, B^0 is 1. So then 38 times 1 is just 38. Let me check that again. If I have, say, 5^0, that's 1, so 38*B^0 would be 38*1=38. Yeah, that makes sense. I don't think there's any trick here. So the answer should be 38.
</think>

The expression $38B^0$ simplifies using the **zero exponent rule**, which states that any non-zero number raised to the power of 0 equals 1. Therefore:

$$
38B^0 = 38 \cdot 1 = 38
$$

In [ ]:
from tqdm import tqdm
import torch

SYSTEM_PROMPT = (
    "You are an expert mathematician. "
    "Solve the problem carefully. "
    "Do not refuse. "
    "End your response with exactly one final answer in the format \\boxed{...}."
)

valid_responses = []

model.eval()

for item in tqdm(valid_rows, desc="Generating validation responses"):
    question = item["question"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    valid_responses.append(response)

Generating validation responses:   4%|▍         | 9/226 [08:03<3:14:22, 53.74s/it]


KeyboardInterrupt: 

In [ ]:
import re
import sys, os
from judger import Judger

judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

results = []

for item, response in zip(valid_rows, valid_responses):
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id": item["id"],
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "correct": correct,
    })

mcq_res = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
print(f"MCQ       : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d} ({acc(mcq_res):.2f}%)")
print(f"Free-form : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d} ({acc(free_res):.2f}%)")
print(f"Overall   : {sum(r['correct'] for r in results):4d} / {len(results):4d} ({acc(results):.2f}%)")
print("=" * 50)

### Google Colab

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv — same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.

In [ ]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 55.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 150.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 85.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 208.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 125.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 73.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 M

Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x7a8aa746d300>
Traceback (most recent call last):
  File "/usr/lib/python3.12/_weakrefset.py", line 39, in _remove
    def _remove(item, selfref=ref(self)):

KeyboardInterrupt: 


     ━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/455.2 MB 287.9 MB/s eta 0:00:02
ERROR: Operation cancelled by user


## 2. Imports & configuration

- `MAX_TOKENS` — **32768** = pub-003 path (default)
- `SUBMISSION_CSV` — `results/submission.csv`
- `DATA_PATH` — `data/private.jsonl`

Prompts are chosen automatically in §5 (no variant knob).

In [ ]:
import csv
import json
import os
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

MAX_TOKENS = 32768

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
DATA_PATH = str(REPO_ROOT / "data/private.jsonl")
SUBMISSION_CSV = str(REPO_ROOT / "results/submission.csv")

_TOKEN_K = MAX_TOKENS // 1024

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"MAX_TOKENS     = {MAX_TOKENS} ({_TOKEN_K}k)")
print(f"SUBMISSION_CSV = {SUBMISSION_CSV}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

REPO_ROOT      = /content/151B_SP26_Competition
MAX_TOKENS     = 32768 (32k)
SUBMISSION_CSV = /content/151B_SP26_Competition/results/submission.csv


## 3. Colab: copy `private.jsonl` from Drive

Upload **`private.jsonl`** to `My Drive/CSE151B/data/private.jsonl` (or change `DRIVE_PRIVATE`). Skip on local clone when `data/private.jsonl` exists.

In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/private.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_PRIVATE = Path("/content/151B_SP26_Competition/data/private.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_PRIVATE.parent.parent
    if not DRIVE_PRIVATE.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_PRIVATE}\nUpload private.jsonl or set DRIVE_PRIVATE."
        )
    #(REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    #dest = REPO_ROOT / "data/private.jsonl"
    #shutil.copy2(DRIVE_PRIVATE, dest)
    #print(f"Copied to {dest.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Load dataset

Private rows have `question`, optional `options`, and `id` — **no** `answer` field.

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |

In [ ]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

Loaded 943 questions  (300 MCQ, 643 free-form) from /content/151B_SP26_Competition/data/private.jsonl

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
} 

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
} 



## 5. Prompt construction

Per question — no global variant switch:

| Case | System prompt |
|------|----------------|
| MCQ | baseline |
| Free-form, 1 `[ANS]` | baseline |
| Free-form, 2+ `[ANS]` | multi-blank (`\boxed{a}, \boxed{b}, ...`, judger-compatible) |

In [ ]:
_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return _MCQ_BASELINE, f"{question}\n\nOptions:\n{opts_text}"
    n_blanks = n_ans_placeholders(question)
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        user = (
            f"{question}\n\n"
            f"The problem has {n_blanks} [ANS] blanks. "
            f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
            f"in order: {example}"
        )
        return _MATH_MULTI_BLANK, user
    return _MATH_BASELINE, question


def prompt_mode(question: str, options: Optional[list]) -> str:
    if options:
        return "mcq/baseline"
    return "multi-blank" if n_ans_placeholders(question) > 1 else "baseline"


for label, item in [
    ("MCQ", mcq_sample),
    ("Free-form (single-blank)", free_sample),
    *( [(f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample)] if multi_sample is not free_sample else [] ),
]:
    mode = prompt_mode(item["question"], item.get("options"))
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} [{mode}] ──")
    print(f"  system: {sys_p[:120]}...")
    print(f"  user (first 300 chars): {usr_p[:300]}...\n")

── MCQ [mcq/baseline] ──
  system: You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer. Outp...
  user (first 300 chars): Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by one percent
E. Decreased by ten percent
F. Halved
G. Unable to determine
H. Doubled
I. Decreased by ...

── Free-form (single-blank) [multi-blank] ──
  system: You are an expert mathematician. Solve the problem step-by-step. For problems with multiple [ANS] placeholders, put each...
  user (first 300 chars): Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS]

The problem has 2 [ANS] blanks. After your reasoning, give 2 comma-separated \boxed{} values in order: \boxed{...}, \boxed{...}...



## 6. Load model with vLLM (A100 profile)

Same **bf16** profile as `full_public.ipynb` — not the starter L4 INT8 path. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).

| Parameter | Value |
|-----------|-------|
| `dtype` | `bfloat16` |
| `max_model_len` | 32768 |
| `enable_prefix_caching` | True |
| `enable_chunked_prefill` | True |

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=32768,
        trust_remote_code=True,
        max_num_seqs=128,
        max_num_batched_tokens=32768,
        enable_chunked_prefill=True,
    )

default_sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=42,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

INFO 05-30 05:29:59 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, c

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

INFO 05-30 05:30:19 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 17.271888 seconds
INFO 05-30 05:30:19 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 159.93 GiB.
INFO 05-30 05:30:19 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-30 05:30:22 [default_loader.py:384] Loading weights took 2.50 seconds
INFO 05-30 05:30:23 [gpu_model_runner.py:4879] Model loading took 7.61 GiB memory and 21.186805 seconds
INFO 05-30 05:30:34 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/0d05882662/rank_0_0/backbone for vLLM's torch.compile
INFO 05-30 05:30:34 [backends.py:1128] Dynamo bytecode transform time: 10.47 s
INFO 05-30 05:30:43 [backends.py:376] Cache the graph of compile range (1, 32768) for later use
INFO 05-30 05:30:50 [backends.py:391] Compiling a graph for compile range (1, 32768) takes 15.36 s
INFO 05-30 05:30:56 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/794492d58e7b8fe2c4af744208da8b4770a4bfaea37acb04b33d5a6381c70e9d/rank_0_0/model
INFO 05-30 05:30:56 [monitor.py:53] torch.compile took 33.08 s in total
INFO 05-30 05:30:56 [monitor.py:81] Initial profiling/warmup run took 0.19 s
INFO 05-30 05:31:08 [gpu_model_run

## 7. Generate responses

Checkpoint: `results/submission.responses.jsonl` (Drive on Colab). Delete checkpoint to regenerate from scratch.

In [ ]:
CHUNK_SIZE = 100

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(SUBMISSION_CSV).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(SUBMISSION_CSV).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    chunk_params = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)
        chunk_params.append(default_sampling_params)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=chunk_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

assert len(completed) == len(data), "Missing ids — checkpoint vs DATA_PATH mismatch"
responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Checkpoint path: /content/151B_SP26_Competition/results/submission.responses.jsonl
Generating 943 remaining / 943 total...


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  800/943  (84.8%)


Rendering prompts:   0%|          | 0/43 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/43 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  900/943  (95.4%)
  943/943  (100.0%)
Done. 943 responses ready.


## 8. Write submission CSV

Uses Python’s `csv` writer so commas and newlines inside `response` are quoted per RFC 4180.

In [ ]:
try:
    csv_out = DRIVE_PROJECT_ROOT / "results" / Path(SUBMISSION_CSV).name
except NameError:
    csv_out = Path(SUBMISSION_CSV)

csv_out.parent.mkdir(parents=True, exist_ok=True)

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    w.writerow(["id", "response"])
    for row in data:
        qid = row["id"]
        w.writerow([qid, completed[qid]])

print(f"Wrote {len(data)} rows to {csv_out.resolve()}")

with open(csv_out, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    n = sum(1 for _ in reader)
assert n == len(data) + 1, f"Expected header + {len(data)} rows, got {n} lines"
print("CSV line count OK (1 header +", len(data), "data rows).")

Wrote 943 rows to /content/151B_SP26_Competition/results/submission.csv
CSV line count OK (1 header + 943 data rows).


## Done

Upload **`submission.csv`** to the course leaderboard workflow. Keep **`submission.responses.jsonl`** as a backup of raw traces.

Pipeline matches **pub-003** (`full_public.ipynb`): 32k tokens, bf16 A100, adaptive multi-blank prompts.

In [ ]:
try:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")
except NameError:
    print("Drive not mounted — skipping runtime termination.")

In [ ]:
!ls -lh /content/151B_SP26_Competition/results

ls: cannot access '/content/151B_SP26_Competition/results': No such file or directory
